# Signature Verification using SVM

This notebook implements signature verification using Support Vector Machine (SVM) with SIFT features and contour-based features.

In [ ]:
# Import necessary libraries
import numpy as np
import os
from os import listdir
import cv2
from PIL import Image
import imagehash
from scipy.cluster.vq import kmeans, vq
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# Define paths
import os
print(f"Current working directory: {os.getcwd()}")

DATA_DIR = "data"
GENUINE_DIR = os.path.join(DATA_DIR, "genuine")
FORGED_DIR = os.path.join(DATA_DIR, "forged")
TEST_DIR = os.path.join(DATA_DIR, "origin")
EXTERNAL_TEST_DIR = "static/LineSweep_Results"
MODEL_PATH = "model.pkl"
SCALER_PATH = "scaler.pkl"

# Number of user groups
NUM_USERS = 29

# SIFT vocabulary size
VOCAB_SIZE = 500

print(f"Model will be saved to: {os.path.abspath(MODEL_PATH)}")
print("Configuration loaded successfully!")

## Import Custom Modules

In [ ]:
# Import custom preprocessing and feature extraction modules
import sys
import importlib

sys.path.insert(0, os.getcwd())

import preproc
import features

print("Custom modules imported successfully!")

## Data Loading and Exploration

In [ ]:
# Load filenames from directories
genuine_filenames = listdir(GENUINE_DIR)
forged_filenames = listdir(FORGED_DIR)

print(f"Total Number of Files in genuine folder: {len(genuine_filenames)}")
print(f"Total Number of Files in forged folder: {len(forged_filenames)}")

# Check if test directory exists
test_filenames = []
if os.path.exists(TEST_DIR):
    test_filenames = listdir(TEST_DIR)
    print(f"Total Number of Files in test folder: {len(test_filenames)}")

# Check if external test directory exists
external_test_filenames = []
if os.path.exists(EXTERNAL_TEST_DIR):
    external_test_filenames = listdir(EXTERNAL_TEST_DIR)
    print(f"Total Number of Files in external test folder: {len(external_test_filenames)}")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    if i < 5:
        img_path = os.path.join(GENUINE_DIR, genuine_filenames[i])
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.set_title(f"Genuine\n{genuine_filenames[i]}")
    else:
        img_path = os.path.join(FORGED_DIR, forged_filenames[i-5])
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.set_title(f"Forged\n{forged_filenames[i-5]}")
    ax.axis('off')

plt.suptitle('Sample Signatures', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Data distribution visualization
fig, ax = plt.subplots(figsize=(8, 5))

categories = ['Genuine', 'Forged', 'Test']
counts = [len(genuine_filenames), len(forged_filenames), len(test_filenames)]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax.bar(categories, counts, color=colors, edgecolor='black', linewidth=1.2)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            str(count), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_xlabel('Category', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('Dataset Distribution', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

## Group Signatures by User

In [ ]:
# Initialize lists for 29 user groups
genuine_features = [[] for _ in range(NUM_USERS)]
forged_features = [[] for _ in range(NUM_USERS)]
test_features = []

# Group genuine signatures by user ID
for name in genuine_filenames:
    signature_id = int(name.split('_')[0][-3:])
    genuine_features[signature_id - 1].append({"name": name})

# Group forged signatures by user ID
for name in forged_filenames:
    signature_id = int(name.split('_')[0][-3:])
    forged_features[signature_id - 1].append({"name": name})

# Group test signatures by user ID
for name in test_filenames:
    signature_id = int(name.split('_')[0][-3:])
    test_features.append({"name": name})

# Count signatures per user
sigs_per_user = len(genuine_features[0])
print(f"Total Genuine Signatures per User: {sigs_per_user}")
print(f"Total Forged Signatures per User: {sigs_per_user}")

In [ ]:
# Visualize signatures per user
fig, ax = plt.subplots(figsize=(12, 5))

users = list(range(1, NUM_USERS + 1))
genuine_counts = [len(genuine_features[i]) for i in range(NUM_USERS)]
forged_counts = [len(forged_features[i]) for i in range(NUM_USERS)]

x = np.arange(len(users))
width = 0.35

bars1 = ax.bar(x - width/2, genuine_counts, width, label='Genuine', color='#2ecc71', edgecolor='black')
bars2 = ax.bar(x + width/2, forged_counts, width, label='Forged', color='#e74c3c', edgecolor='black')

ax.set_xlabel('User ID', fontsize=12)
ax.set_ylabel('Number of Signatures', fontsize=12)
ax.set_title('Signatures per User', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(users)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Feature Extraction Functions

In [ ]:
def preprocess_image(path, display=False):
    """Preprocess the image"""
    return preproc.preproc(path, display=display)

def extract_sift_features(im, path, display=False):
    """Extract SIFT features from image"""
    raw_image = cv2.imread(path)
    try:
        sift_detector = cv2.xfeatures2d.SIFT_create()
    except AttributeError:
        sift_detector = cv2.SIFT_create()
    kp, des = sift_detector.detectAndCompute(im, None)
    
    if display:
        cv2.drawKeypoints(im, kp, raw_image)
        cv2.imshow('sift_keypoints.jpg', cv2.resize(raw_image, (0, 0), fx=3, fy=3))
        cv2.waitKey()
    
    return (path, des)

def extract_contour_features(image_path):
    """Extract contour-based features from image"""
    preprocessed_image = preprocess_image(image_path)
    phash = imagehash.phash(Image.open(image_path))
    
    aspect_ratio, bounding_rect_area, convex_hull_area, contours_area = \
        features.get_contour_features(preprocessed_image.copy(), display=False)
    
    ratio = features.Ratio(preprocessed_image.copy())
    centroid_0, centroid_1 = features.Centroid(preprocessed_image.copy())
    eccentricity, solidity = features.EccentricitySolidity(preprocessed_image.copy())
    (skewness_0, skewness_1), (kurtosis_0, kurtosis_1) = features.SkewKurtosis(preprocessed_image.copy())
    
    contour_features = [
        aspect_ratio,
        convex_hull_area / bounding_rect_area if bounding_rect_area > 0 else 0,
        contours_area / bounding_rect_area if bounding_rect_area > 0 else 0,
        ratio,
        centroid_0,
        centroid_1,
        eccentricity,
        solidity,
        skewness_0,
        skewness_1,
        kurtosis_0,
        kurtosis_1
    ]
    
    return contour_features

print("Feature extraction functions defined!")

## Image Preprocessing Visualization

In [ ]:
# Show preprocessing steps for a sample image
sample_img_path = os.path.join(GENUINE_DIR, genuine_filenames[0])
original_img = plt.imread(sample_img_path)
preprocessed_img = preprocess_image(sample_img_path)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(original_img)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(preprocessed_img, cmap='gray')
axes[1].set_title('Preprocessed Image', fontsize=12)
axes[1].axis('off')

plt.suptitle('Image Preprocessing Steps', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Train the SVM Model

In [ ]:
# Collect all features for training
all_train_features = []
all_train_labels = []
all_test_features = []
all_test_labels = []

print("Processing training data...")

for i in range(NUM_USERS):
    des_list = []
    im_contour_features = []
    
    # Process genuine images
    for im in genuine_features[i]:
        image_path = os.path.join(GENUINE_DIR, im['name'])
        try:
            contour_feats = extract_contour_features(image_path)
            im_contour_features.append(contour_feats)
            des_list.append(extract_sift_features(preprocess_image(image_path), image_path))
        except Exception as e:
            print(f"Error processing {im['name']}: {e}")
            continue
    
    # Process forged images
    for im in forged_features[i]:
        image_path = os.path.join(FORGED_DIR, im['name'])
        try:
            contour_feats = extract_contour_features(image_path)
            im_contour_features.append(contour_feats)
            des_list.append(extract_sift_features(preprocess_image(image_path), image_path))
        except Exception as e:
            print(f"Error processing {im['name']}: {e}")
            continue
    
    # Create vocabulary
    descriptors = des_list[0][1]
    for path, descriptor in des_list[1:]:
        if descriptor is not None:
            try:
                descriptors = np.vstack((descriptors, descriptor))
            except:
                pass
    
    desired_k = VOCAB_SIZE
    effective_k = desired_k
    if descriptors.shape[0] < desired_k:
        effective_k = descriptors.shape[0]
    
    if effective_k > 0:
        voc, variance = kmeans(descriptors, effective_k, 1)
    
    n_images = len(genuine_features[i]) + len(forged_features[i])
    im_features = np.zeros((n_images, desired_k + 12), "float32")
    
    for ii in range(n_images):
        if des_list[ii][1] is not None:
            try:
                words, distance = vq(des_list[ii][1], voc)
                for w in words:
                    im_features[ii][w] += 1
            except:
                pass
        
        for j in range(12):
            im_features[ii][desired_k + j] = im_contour_features[ii][j]
    
    # Split into training (3 train, 2 test per class)
    train_genuine = im_features[0:3]
    test_genuine = im_features[3:5]
    train_forged = im_features[5:8]
    test_forged = im_features[8:10]
    
    all_train_features.append(train_genuine)
    all_train_features.append(train_forged)
    all_train_labels.extend([2] * len(train_genuine))  # 2 = genuine
    all_train_labels.extend([1] * len(train_forged))   # 1 = forged
    
    all_test_features.append(test_genuine)
    all_test_features.append(test_forged)
    all_test_labels.extend([2] * len(test_genuine))
    all_test_labels.extend([1] * len(test_forged))
    
    print(f"User {i+1}: Processed {n_images} images")

# Combine all features
X_train = np.vstack(all_train_features)
y_train = np.array(all_train_labels)
X_test = np.vstack(all_test_features)
y_test = np.array(all_test_labels)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n=== Training Data Summary ===")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Feature dimension: {X_train.shape[1]}")
print(f"Label distribution - Genuine: {y_train.tolist().count(2)}, Forged: {y_train.tolist().count(1)}")

In [ ]:
# Train the SVM model
print("\nTraining SVM model...")
clf = LinearSVC()
clf.fit(X_train_scaled, y_train)

# Save model and scaler directly to current directory
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(clf, f)

with open(SCALER_PATH, 'wb') as f:
    pickle.dump(scaler, f)

print(f"\nModel trained and saved to: {os.path.abspath(MODEL_PATH)}")
print(f"Scaler saved to:            {os.path.abspath(SCALER_PATH)}")

## Why Two Files? (model.pkl and scaler.pkl)

**model.pkl** - The trained SVM classifier that makes predictions

**scaler.pkl** - The StandardScaler that normalizes features using the same statistics learned from training data. Without this, test data would be scaled differently, leading to incorrect predictions.

## Model Evaluation on Test Set

In [ ]:
# Make predictions on test set
y_pred = clf.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"=== Test Set Results ===")
print(f"Accuracy: {accuracy * 100:.2f}%")

# Separate genuine vs forged accuracy
genuine_mask = y_test == 2
forged_mask  = y_test == 1
genuine_acc = accuracy_score(y_test[genuine_mask], y_pred[genuine_mask])
forged_acc  = accuracy_score(y_test[forged_mask],  y_pred[forged_mask])
print(f"Genuine accuracy: {genuine_acc * 100:.2f}%")
print(f"Forged  accuracy: {forged_acc  * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Forged (1)', 'Genuine (2)']))

In [ ]:
# Confusion Matrix Visualization
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Forged', 'Genuine'],
            yticklabels=['Forged', 'Genuine'],
            ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Per-user accuracy visualization
# Test layout: for user i → indices [i*4, i*4+1] = genuine, [i*4+2, i*4+3] = forged
fig, ax = plt.subplots(figsize=(14, 5))

user_accuracies = []
for i in range(NUM_USERS):
    base = i * 4
    preds_gen = clf.predict(X_test_scaled[base:base+2])     # 2 genuine samples
    preds_for = clf.predict(X_test_scaled[base+2:base+4])   # 2 forged samples

    correct = sum(1 for p in preds_gen if p == 2) + sum(1 for p in preds_for if p == 1)
    user_accuracies.append(correct / 4 * 100)

colors = ['#2ecc71' if acc >= 50 else '#e74c3c' for acc in user_accuracies]
bars = ax.bar(range(1, NUM_USERS + 1), user_accuracies, color=colors, edgecolor='black')

ax.axhline(y=50, color='orange', linestyle='--', linewidth=2, label='50% threshold')
ax.set_xlabel('User ID', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-User Test Accuracy', fontsize=14, fontweight='bold')
ax.set_ylim(0, 110)
ax.legend()

for bar, acc in zip(bars, user_accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.0f}%', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

print(f"Average per-user accuracy: {np.mean(user_accuracies):.1f}%")
print(f"Users with 100% accuracy:  {sum(1 for a in user_accuracies if a == 100)}/{NUM_USERS}")

## Feature Importance Analysis

In [ ]:
# Visualize feature statistics (using first 12 contour features)
feature_names = ['Aspect Ratio', 'Hull/Bound Area', 'Contour/Bound Area', 
                 'Ratio', 'Centroid 0', 'Centroid 1', 'Eccentricity', 
                 'Solidity', 'Skewness 0', 'Skewness 1', 'Kurtosis 0', 'Kurtosis 1']

train_genuine_features = X_train[y_train == 2][:, VOCAB_SIZE:]
train_forged_features = X_train[y_train == 1][:, VOCAB_SIZE:]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, (name, ax) in enumerate(zip(feature_names, axes)):
    ax.hist(train_genuine_features[:, i], bins=20, alpha=0.6, label='Genuine', color='#2ecc71')
    ax.hist(train_forged_features[:, i], bins=20, alpha=0.6, label='Forged', color='#e74c3c')
    ax.set_title(name, fontsize=10)
    ax.legend(fontsize=8)
    
plt.suptitle('Feature Distribution: Genuine vs Forged', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Model Summary

In [ ]:
# Model Summary
print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(f"\nModel Type     : LinearSVC (Linear Support Vector Classifier)")
print(f"Feature Dim    : {X_train.shape[1]}")
print(f"  SIFT BoVW    : {VOCAB_SIZE} visual words")
print(f"  Geometric    : 12 contour features")
print(f"\nTraining set   : {len(X_train)} samples  ({y_train.tolist().count(2)} genuine / {y_train.tolist().count(1)} forged)")
print(f"Test set       : {len(X_test)} samples  ({y_test.tolist().count(2)} genuine / {y_test.tolist().count(1)} forged)")
print(f"\nTest Accuracy  : {accuracy * 100:.2f}%")
print(f"Genuine Acc    : {genuine_acc * 100:.2f}%")
print(f"Forged  Acc    : {forged_acc  * 100:.2f}%")
print(f"\nFiles saved    :")
print(f"  {MODEL_PATH}")
print(f"  {SCALER_PATH}")
print("=" * 60)

In [ ]:
# Visual summary
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Dataset split
ax1 = axes[0]
sizes = [len(X_train), len(X_test)]
labels = ['Training', 'Test']
colors = ['#3498db', '#e74c3c']
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('Dataset Split', fontsize=12, fontweight='bold')

# 2. Class distribution
ax2 = axes[1]
train_counts = [y_train.tolist().count(1), y_train.tolist().count(2)]
test_counts = [y_test.tolist().count(1), y_test.tolist().count(2)]
x = np.arange(2)
width = 0.35
ax2.bar(x - width/2, train_counts, width, label='Training', color='#3498db')
ax2.bar(x + width/2, test_counts, width, label='Test', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(['Forged', 'Genuine'])
ax2.set_ylabel('Count')
ax2.set_title('Class Distribution', fontsize=12, fontweight='bold')
ax2.legend()

# 3. Accuracy
ax3 = axes[2]
accuracies = [accuracy * 100, 100 - accuracy * 100]
ax3.pie(accuracies, labels=['Correct', 'Incorrect'], colors=['#2ecc71', '#e74c3c'], 
        autopct='%1.1f%%', startangle=90)
ax3.set_title(f'Test Accuracy: {accuracy*100:.1f}%', fontsize=12, fontweight='bold')

plt.suptitle('Model Training Summary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Quick Verification Function

In [ ]:
def verify_signature(image_path, model, scaler):
    """
    Verify a single signature image using the trained model.

    Parameters:
    - image_path: Path to the signature image
    - model: Trained LinearSVC
    - scaler: Fitted StandardScaler

    Returns:
    - "Genuine" or "Forged"
    """
    try:
        contour_feats = extract_contour_features(image_path)
        preprocessed = preprocess_image(image_path)
        try:
            sift_detector = cv2.xfeatures2d.SIFT_create()
        except AttributeError:
            sift_detector = cv2.SIFT_create()
        kp, des = sift_detector.detectAndCompute(preprocessed, None)

        feat_vec = np.zeros((1, VOCAB_SIZE + 12), "float32")
        for j in range(12):
            feat_vec[0][VOCAB_SIZE + j] = contour_feats[j]

        feat_vec_scaled = scaler.transform(feat_vec)
        prediction = model.predict(feat_vec_scaled)

        return "Genuine" if prediction[0] == 2 else "Forged"
    except Exception as e:
        return f"Error: {str(e)}"


print("verify_signature() ready.")
print("Usage: result = verify_signature('path/to/signature.png', clf, scaler)")

# Quick test on a training image
sample_path = os.path.join(GENUINE_DIR, genuine_filenames[0])
result = verify_signature(sample_path, clf, scaler)
print(f"\nSample test on '{genuine_filenames[0]}': {result}  (expected: Genuine)")